In [3]:
import pdfplumber
import os
import pathlib
from pathlib import Path

import sys
sys.path.insert(0, "/workspaces/Capstone-AI-Engineering-Buildcamp")

from pathlib import Path
from store.base import Chunk

pdf_path = "../data/raw_pdfs/Edelweiss Factsheet June - 26_19062026_070749_PM.pdf"

with pdfplumber.open(pdf_path) as pdf:
    first_page = pdf.pages[7]
    text = first_page.extract_text()
    print(text)

Edelweiss Mid Cap Fund
An open ended equity scheme predominantly investing in mid cap
stocks
Data as on May 31, 2026
About the Scheme
Inception Date 26-Dec-07
A midcap focused fund that primarily invests 65% in midcap companies with strong business
Benchmark Nifty Midcap 150 TRI fundamentals promising good earnings and growth opportunities.
NAV Our “FAIR” investment framework helps in identifying robust and clean businesses available at
acceptable prices without being biased toward either of the investing styles.
Direct Plan IDCW Option : ₹90.75
Direct Plan Growth Option : ₹124.46
Regular Plan IDCW Option : ₹60.80 Top 30 Holdings
Regular Plan Growth Option : ₹105.46
Company Name Allocation Company Name Allocation
Expense Ratio1
Multi Commodity Exchange Of India Ltd 3.03% AU Small Finance Bank Ltd 1.55%
BER / TER (Regular Plan) :1.43%/1.67% BSE Ltd 2.94% Indus Towers Ltd 1.53%
BER / TER ( Direct Plan) :0.41%/0.48% Federal Bank Ltd 2.90% Bharat Forge Ltd 1.53%
Fund Size Fortis Healthcare

In [5]:
with pdfplumber.open(pdf_path) as pdf:
    print(f"total pages: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages[:5]):
        text = page.extract_text() or ""
        print(f"Page {i+1} has {len(text)} words")

total pages: 193
Page 1 has 821 words
Page 2 has 52 words
Page 3 has 1678 words
Page 4 has 1619 words
Page 5 has 3435 words


In [6]:
for line in text.split("\n"):
    print(line.strip())


Expert Speaks
June quarter most challenging
Just last week, I bought a phone for my cousin using a quick commerce app. We were casually chatting The June
quarter is likely to be the most challenging period for Indian companies as the economic impact of elevated fuel
prices and a delayed monsoon starts showing up in financial results. The fundamentals are expected to weaken
before improving, with June likely witnessing the sharpest impact of the West Asia conflict on corporate earnings.
Some spillover effects may also continue into the September quarter, depending on how geopolitical developments
evolve.
Despite these concerns, a significant portion of the weakness is believed to have already been priced into equity
markets. Broader markets have corrected between 5% and 10% from their peaks, broadly reflecting expectations of
a weak June quarter. Any deterioration beyond current expectations, particularly extending into the September
quarter, would be a fresh concern for investors.
Unde

In [7]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[7]  # try page 6 — factsheets usually have tables there
    tables = page.extract_tables()
    print(f"Tables found: {len(tables)}")
    if tables:
        print(tables[5])  # print first table

Tables found: 10
[['Company Name', 'Weights C\n(%)', 'ontribution\n(%)']]


In [8]:
tables[5]

[['Company Name', 'Weights C\n(%)', 'ontribution\n(%)']]

In [7]:
# def table_to_text(table: list[list]) -> str:
#     if not table:
#         return ""
#     headers = table[0]
#     rows = table[1:]
#     lines = []
#     for row in rows:
#         parts = [f"{headers[i]}: {row[i]}" for i in range(len(headers)) if row[i]]
#         lines.append(", ".join(parts))
#     return "\n".join(lines)

# # test it
# table_text = table_to_text(tables[5])
# print(table_text)

In [9]:
def table_to_text(table: list[list]) -> str:
    if not table:
        return ""
    lines = []
    for row in table:
       lines.append(" | ".join(cell or "" for cell in row))

    return "\n".join(lines)

# test it
table_text = table_to_text(tables[5])
type(table_text)

str

In [10]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[7]
    lines = page.extract_text_lines()
    print(lines[1]["chars"])


[{'matrix': (8.04, 0.0, 0.0, 8.04, 28.1249, 792.3324), 'fontname': 'QSPGTC+Roboto-Regular', 'adv': 0.632, 'upright': True, 'x0': 28.1249, 'y0': 790.15356, 'x1': 33.20618, 'y1': 798.19356, 'width': 5.081280000000003, 'height': 8.040000000000077, 'size': 8.040000000000077, 'mcid': 683, 'tag': 'P', 'object_type': 'char', 'page_number': 8, 'ncs': 'DeviceGray', 'text': 'A', 'stroking_color': (0.553, 0.776, 0.247), 'non_stroking_color': (1.0,), 'top': 43.72643999999991, 'bottom': 51.76643999999999, 'doctop': 5941.21644}, {'matrix': (8.04, 0.0, 0.0, 8.04, 33.165176, 792.3324), 'fontname': 'QSPGTC+Roboto-Regular', 'adv': 0.5680000000000001, 'upright': True, 'x0': 33.165176, 'y0': 790.15356, 'x1': 37.731896000000006, 'y1': 798.19356, 'width': 4.566720000000004, 'height': 8.040000000000077, 'size': 8.040000000000077, 'mcid': 683, 'tag': 'P', 'object_type': 'char', 'page_number': 8, 'ncs': 'DeviceGray', 'text': 'n', 'stroking_color': (0.553, 0.776, 0.247), 'non_stroking_color': (1.0,), 'top': 43.

In [ ]:
def detect_scheme_name(page):
    lines = page.extract_text_lines()
    if not lines:
        return None

    def font_size(line):
        return max(c["size"] for c in line["chars"])

    candidates = [
        ln for ln in lines
        if ("fund" in ln["text"].lower() or "scheme" in ln["text"].lower())
        and len(ln["text"].strip()) < 80
    ]
    
    if not candidates:
        return None
    
    best = max(candidates, key=font_size)
    text = best["text"].strip()

    idx = text.lower().find("fund")
    
    if idx != -1:
        text = text[: idx + len("fund")]
    
    return text.strip()


In [14]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[7]
    print(detect_scheme_name(page))

[{'text': 'Edelweiss Mid Cap Fund', 'x0': 28.1249, 'top': 23.72569999999996, 'x1': 223.56890000000004, 'bottom': 41.72569999999996, 'chars': [{'matrix': (18.0, 0.0, 0.0, 18.0, 28.1249, 805.0723), 'fontname': 'HWHJXU+Roboto-Bold', 'adv': 0.556, 'upright': True, 'x0': 28.1249, 'y0': 800.1943, 'x1': 38.1329, 'y1': 818.1943, 'width': 10.008, 'height': 18.0, 'size': 18.0, 'mcid': 676, 'tag': 'P', 'object_type': 'char', 'page_number': 8, 'ncs': 'DeviceGray', 'text': 'E', 'stroking_color': (0.553, 0.776, 0.247), 'non_stroking_color': (1.0,), 'top': 23.72569999999996, 'bottom': 41.72569999999996, 'doctop': 5921.2157}, {'matrix': (18.0, 0.0, 0.0, 18.0, 37.965500000000006, 805.0723), 'fontname': 'HWHJXU+Roboto-Bold', 'adv': 0.5640000000000001, 'upright': True, 'x0': 37.965500000000006, 'y0': 800.1943, 'x1': 48.11750000000001, 'y1': 818.1943, 'width': 10.152000000000001, 'height': 18.0, 'size': 18.0, 'mcid': 676, 'tag': 'P', 'object_type': 'char', 'page_number': 8, 'ncs': 'DeviceGray', 'text': 'd

In [7]:
def extract_page_content(page) -> str:
    # Step 1: extract plain text
    plain_text = page.extract_text() or ""
    
    # Step 2: extract tables and convert each to text using table_to_text()
    tables = page.extract_tables()
    
    table_text = []
    for table in tables:
        table_text.append(table_to_text(table))
    
    # Step 3: combine both — plain text first, then table text below it
    combined_text = plain_text + '\n' + '\n'.join(table_text)
    
    # Step 4: return the combined string
    return combined_text

In [8]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[7]  # try page 6 — factsheets usually have tables there
    # tables = page.extract_tables()

    # print(table_to_text(tables[1]))
    combined_text = extract_page_content(page)

In [9]:
print(combined_text)

Investment style Scan to Invest Now
KOTAK LARGE CAP FUND Value GARP Growth Size
Large cap fund - An open-ended equity scheme predominantly investing in large cap stocks Large
Medium
Investment Objective: To generate capital appreciation from a portfolio of predominantly
equity and equity related securities falling under the category of large Cap companies. Small
However, there is no assurance that the objective of the scheme will be achieved. GARP - Growth at a Reasonable Price
Fund Manager*: Mr. Rohit Tandon PORTFOLIO
Issuer/Instrument % to Net Assets Issuer/Instrument % to Net Assets
AAUM: `10,574.59 crs
Equity & Equity related Ultratech Cement Ltd. 2.09
Banks 22.17 India Cements Ltd. 0.36
AUM: `10,516.39 crs ICICI Bank Ltd. 6.98 Ferrous Metals 2.26
HDFC Bank Ltd. 6.25 Tata Steel Ltd. 2.26
STATE BANK OF INDIA 4.20 Chemicals and Petrochemicals 2.13
Benchmark***: Nifty 100 TRI (Tier 1), Axis Bank Ltd. 3.25 SOLAR INDUSTRIES INDIA LIMITED 2.13
Nifty 50 TRI (Tier 2) KOTAK MAHINDRA BANK LT

In [10]:
def chunk_text(text: str, page: int, fund_name: str, 
               source_file: str, chunk_size: int = 500, 
               overlap: int = 50) -> list[Chunk]:
    # Step 1: split text into words
    split_text = text.split()
        
    start = 0
    end = start + chunk_size
    chunk_list = []

    while start < len(split_text):
    # Step 2:
        window = split_text[start:end]

    # Step 3: for each window, join words back into a string
        content = " ".join(window)
    # Step 4: create a chunk_id: f"{fund_name}_p{page}_{index}"
        chunk_id = f"{Path(source_file).stem}_p{page}_c{start+1}"
    # Step 5: create a Chunk object (leave embedding as empty list [] for now)
        chunk = Chunk(chunk_id=chunk_id,
                      text = content,
                      embedding=[],
                      fund_name=fund_name,
                      source_file=source_file,
                      page=page,
                      metadata={})
        
        chunk_list.append(chunk)
        
        start += chunk_size - overlap
        end = start + chunk_size

    # Step 6: return the list of Chunks
    return chunk_list

In [11]:
test = chunk_text(combined_text, 7, "Kotak MF", source_file='KotakMFFactsheetMay2026.pdf')
# test
test

[Chunk(chunk_id='KotakMFFactsheetMay2026_p7_c1', text='Investment style Scan to Invest Now KOTAK LARGE CAP FUND Value GARP Growth Size Large cap fund - An open-ended equity scheme predominantly investing in large cap stocks Large Medium Investment Objective: To generate capital appreciation from a portfolio of predominantly equity and equity related securities falling under the category of large Cap companies. Small However, there is no assurance that the objective of the scheme will be achieved. GARP - Growth at a Reasonable Price Fund Manager*: Mr. Rohit Tandon PORTFOLIO Issuer/Instrument % to Net Assets Issuer/Instrument % to Net Assets AAUM: `10,574.59 crs Equity & Equity related Ultratech Cement Ltd. 2.09 Banks 22.17 India Cements Ltd. 0.36 AUM: `10,516.39 crs ICICI Bank Ltd. 6.98 Ferrous Metals 2.26 HDFC Bank Ltd. 6.25 Tata Steel Ltd. 2.26 STATE BANK OF INDIA 4.20 Chemicals and Petrochemicals 2.13 Benchmark***: Nifty 100 TRI (Tier 1), Axis Bank Ltd. 3.25 SOLAR INDUSTRIES INDIA LI

In [12]:
from sentence_transformers import SentenceTransformer

# Step 1: load the model (downloads on first run, cached after)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2: get your chunks from chunk_text()
chunks = chunk_text(combined_text, 7, "Kotak MF", source_file="KotakMFFactsheetMay2026.pdf")

# Step 3: extract just the text from each chunk into a plain list
texts = [c.text for c in chunks]

# Step 4: encode — model takes a list of strings, returns a numpy array (n_chunks x 384)
embeddings = model.encode(texts)

# Step 5: assign each embedding back to its chunk
for i, chunk in enumerate(chunks):
    chunk.embedding = embeddings[i].tolist()   # hint: embeddings[i], but convert to list

# Step 6: verify — print the embedding length of the first chunk
print(len(chunks[0].embedding))   # should print 384

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

384


In [13]:
import chromadb
from chromadb.config import Settings

# Step 1: create an in-memory client (no persistence — safe for notebook experiments)
client = chromadb.EphemeralClient()

# Step 2: create a collection with cosine similarity
collection = client.get_or_create_collection(
    name="test_chunks",
    metadata={"hnsw:space": "cosine"}
)

# Step 3: upsert your chunks into the collection
# collection.upsert() takes 4 parallel lists:
#   ids, embeddings, documents, metadatas
collection.upsert(
    ids=[c.chunk_id for c in chunks],        # [c.chunk_id for c in chunks]
    embeddings=[c.embedding for c in chunks], # [c.embedding for c in chunks]
    documents=[c.text for c in chunks], # [c.text for c in chunks]
    metadatas= [{"fund_name":c.fund_name,"source_file":c.source_file,"page":c.page} for c in chunks]  # list of dicts — one per chunk with fund_name, source_file, page
)

# Step 4: verify how many got stored
print(collection.count())

# Step 5: search — encode a query and find the top 3 most similar chunks
query = "What is the expense ratio?"
query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["metadatas", "embeddings","documents"]
)

# # Step 6: print the top results
# for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
#     print(meta)
#     print(doc[:200])
#     print("---")

5


In [ ]:
results

KeyError: 0

In [2]:
chunks = "abc"
if chunks:
    print("true")
else:
    print("false")

print(not(chunks))

true
False


In [6]:
d = {"a":1,"a":32,"b":2,"c":3}
{**d}

{'a': 32, 'b': 2, 'c': 3}

In [10]:
a = tuple([1,2])
print(a)

(1, 2)


In [ ]:
parse (page by page) -> extract text+ table -> chunk -> embed -> store

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3234942660.py, line 1)